In [2]:
"""
Logistic Regression from Scratch (NumPy)
=========================================
This module implements a Binary Logistic Regression classifier from first principles.

Mathematical Overview:
----------------------
1. Forward Pass (Hypothesis Function):
   z = X @ w + b
   ŷ = σ(z) = 1 / (1 + exp(-z))

2. Loss Function (Binary Cross-Entropy / Log Loss):
   J(w, b) = - (1/m) * ∑ [ y * log(ŷ) + (1 - y) * log(1 - ŷ) ]

3. Optimization (Gradient Descent):
   ∂J/∂w = (1/m) * X^T @ (ŷ - y)
   ∂J/∂b = (1/m) * ∑ (ŷ - y)

   w_new = w - α * (∂J/∂w)
   b_new = b - α * (∂J/∂b)

"""

import numpy as np


class LogisticRegressionScratch:
    """
    Binary Logistic Regression classifier built using NumPy.

    Parameters:
    -----------
    learning_rate : float, default=0.01
        The step size (α) used for gradient descent parameter updates.
    n_iters : int, default=1000
        Number of iterations/epochs to run gradient descent.

    Attributes:
    -----------
    weights : np.ndarray of shape (n_features,)
        Learned weights for input features.
    bias : float
        Learned bias (intercept) term.
    loss_history : list of float
        Stores binary cross-entropy loss value at each iteration for diagnostics.
    """

    def __init__(self, learning_rate: float = 0.01, n_iters: int = 1000):
        self.lr = learning_rate
        self.n_iters = n_iters
        self.weights = None
        self.bias = None
        self.loss_history = []

    def _sigmoid(self, z: np.ndarray) -> np.ndarray:
        """
        Computes the Sigmoid (logistic) activation function.

        Derivation / Property:
        ----------------------
        σ(z) = 1 / (1 + e^(-z))

        Maps any real-valued scalar/matrix to the range (0, 1), representing probability.
        We clip 'z' to prevent numerical overflow in np.exp(-z).
        """
        z_clipped = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z_clipped))

    def _compute_loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """
        Computes the Binary Cross-Entropy Loss (Log Loss).

        Mathematical Formula:
        ---------------------
        J(w, b) = - (1/m) * ∑ [ y * log(ŷ) + (1 - y) * log(1 - ŷ) ]

        Note: Epsilon (1e-15) is added to avoid log(0) undefined errors.
        """
        m = y_true.shape[0]
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        loss = - (1 / m) * np.sum(
            y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
        )
        return loss

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LogisticRegressionScratch":
        """
        Fit the model according to the given training data using Gradient Descent.

        Derivation of Gradients:
        ------------------------
        1. Chain Rule for Loss w.r.t weights (w_j):
           ∂J/∂w_j = (∂J/∂ŷ) * (∂ŷ/∂z) * (∂z/∂w_j)

        2. Step-by-Step Partial Derivatives:
           a) ∂J/∂ŷ = - (y / ŷ) + ((1 - y) / (1 - ŷ))
                    = (ŷ - y) / [ ŷ * (1 - ŷ) ]

           b) ∂ŷ/∂z = ∂/∂z [ 1 / (1 + e^-z) ]
                    = e^-z / (1 + e^-z)^2
                    = (1 / (1 + e^-z)) * (e^-z / (1 + e^-z))
                    = ŷ * (1 - ŷ)

           c) ∂z/∂w_j = ∂/∂w_j [ w_1*x_1 + ... + w_j*x_j + b ]
                      = x_j

        3. Multiplying together (Chain Rule):
           ∂J/∂w_j = [ (ŷ - y) / (ŷ * (1 - ŷ)) ] * [ ŷ * (1 - ŷ) ] * x_j
                   = (ŷ - y) * x_j

        4. Vectorized Gradient over all 'm' samples:
           ∂J/∂w = (1 / m) * X^T @ (ŷ - y)
           ∂J/∂b = (1 / m) * ∑ (ŷ - y)
        """
        n_samples, n_features = X.shape

        # Initialize parameters: zero initialization is safe for Logistic Regression
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        self.loss_history = []

        # Optimization loop using Batch Gradient Descent
        for _ in range(self.n_iters):
            # Step 1: Linear combination (z = Xw + b)
            linear_model = np.dot(X, self.weights) + self.bias

            # Step 2: Pass through Sigmoid activation to get probabilities (ŷ)
            y_predicted = self._sigmoid(linear_model)

            # Step 3: Compute gradients via vectorization
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            # Step 4: Update weights and bias (Gradient Descent Rule)
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            # Step 5: Track loss for evaluation/plotting
            current_loss = self._compute_loss(y, y_predicted)
            self.loss_history.append(current_loss)

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """
        Predict probability estimates for input samples.

        Returns:
        --------
        np.ndarray of shape (n_samples,): Probabilities for class 1.
        """
        linear_model = np.dot(X, self.weights) + self.bias
        return self._sigmoid(linear_model)

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        """
        Predict binary class labels for input samples.

        Parameters:
        -----------
        X : np.ndarray of shape (n_samples, n_features)
        threshold : float, default=0.5
            Probability threshold to decide positive class (1).

        Returns:
        --------
        np.ndarray of shape (n_samples,): Predicted binary classes (0 or 1).
        """
        y_predicted_cls = self.predict_proba(X)
        return np.where(y_predicted_cls >= threshold, 1, 0)


# =====================================================================
# Example Usage / Test Driver (Ideal for README.md demonstrate section)
# =====================================================================
if __name__ == "__main__":
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, classification_report

    # 1. Generate Synthetic Binary Classification Dataset
    X, y = make_classification(
        n_samples=1000,
        n_features=5,
        n_classes=2,
        random_state=42
    )

    # 2. Train / Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # 3. Instantiate & Train Model
    clf = LogisticRegressionScratch(learning_rate=0.1, n_iters=1000)
    clf.fit(X_train, y_train)

    # 4. Make Predictions
    predictions = clf.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    # 5. Display Results
    print(f"Model Training Complete!")
    print(f"Final Loss: {clf.loss_history[-1]:.4f}")
    print(f"Test Accuracy: {accuracy * 100:.2f}%\n")
    print("Classification Report:")
    print(classification_report(y_test, predictions))

Model Training Complete!
Final Loss: 0.3514
Test Accuracy: 88.00%

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.92      0.88        97
           1       0.92      0.84      0.88       103

    accuracy                           0.88       200
   macro avg       0.88      0.88      0.88       200
weighted avg       0.88      0.88      0.88       200



In [5]:
"""
Logistic Regression via Newton's Method (NumPy)
==============================================
This module implements Binary Logistic Regression optimized using Newton's Method
(Newton-Raphson Optimization / Iterative Reweighted Least Squares).

Mathematical Overview:
----------------------
1. Objective / Loss Function (Binary Cross-Entropy):
   J(w) = - (1/m) * ∑ [ y * log(ŷ) + (1 - y) * log(1 - ŷ) ]

2. Gradient (Vector / First Derivative):
   g = ∇J(w) = (1/m) * X^T @ (ŷ - y)

3. Hessian Matrix (Second Derivative):
   H = ∇²J(w) = (1/m) * X^T @ S @ X
   where S is an (m x m) diagonal matrix: S_ii = ŷ_i * (1 - ŷ_i)

4. Newton-Raphson Update Rule:
   w_new = w - H^(-1) @ g

   Unlike First-Order Gradient Descent (which uses a constant learning rate α),
   Newton's Method uses curvature (the Hessian) to compute optimal step sizes,
   typically converging in very few iterations (often 5 to 15).

"""

import numpy as np


class LogisticRegressionNewton:
    """
    Binary Logistic Regression classifier optimized with Newton's Method.

    Parameters:
    -----------
    n_iters : int, default=20
        Max iterations. Newton's Method converges super-linearly/quadratically,
        requiring far fewer iterations than standard Gradient Descent.
    tol : float, default=1e-6
        Convergence tolerance for parameter updates (||w_new - w_old|| < tol).
    l2_reg : float, default=1e-5
        Small L2 regularization added to the Hessian matrix (H + λI)
        to ensure invertibility and stability (Ridge regularization).

    Attributes:
    -----------
    weights : np.ndarray of shape (n_features + 1,)
        Learned weights including the absorbed bias term at index 0.
    loss_history : list of float
        Binary cross-entropy loss recorded at each step.
    """

    def __init__(self, n_iters: int = 20, tol: float = 1e-6, l2_reg: float = 1e-5):
        self.n_iters = n_iters
        self.tol = tol
        self.l2_reg = l2_reg
        self.weights = None
        self.loss_history = []

    def _sigmoid(self, z: np.ndarray) -> np.ndarray:
        """
        Computes the Sigmoid activation function.
        σ(z) = 1 / (1 + e^(-z))
        """
        z_clipped = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z_clipped))

    def _compute_loss(self, y_true: np.ndarray, y_pred: np.ndarray) -> float:
        """Computes Binary Cross-Entropy Loss."""
        m = y_true.shape[0]
        epsilon = 1e-15
        y_pred = np.clip(y_pred, epsilon, 1 - epsilon)
        return - (1 / m) * np.sum(
            y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred)
        )

    def fit(self, X: np.ndarray, y: np.ndarray) -> "LogisticRegressionNewton":
        """
        Fit the model using Newton-Raphson Optimization.

        Derivation of Gradient and Hessian:
        ----------------------------------
        1. Hypothesis with Bias trick:
           X_bias = [1, X]  (Shape: m x (n + 1))
           z = X_bias @ w
           ŷ = σ(z)

        2. First Derivative (Gradient Vector, g):
           g_j = ∂J/∂w_j = (1/m) * ∑ (ŷ_i - y_i) * x_ij
           g = (1/m) * X_bias^T @ (ŷ - y)              [Shape: (n+1) x 1]

        3. Second Derivative (Hessian Matrix, H):
           H_jk = ∂²J / (∂w_j ∂w_k)
                = (1/m) * ∑ x_ij * ∂/∂w_k [ σ(z_i) ]
                = (1/m) * ∑ x_ij * [ σ(z_i) * (1 - σ(z_i)) ] * x_ik
                = (1/m) * ∑ x_ij * ŷ_i * (1 - ŷ_i) * x_ik

           Matrix Form:
           H = (1/m) * X_bias^T @ S @ X_bias           [Shape: (n+1) x (n+1)]
           where S = diag(ŷ_1*(1 - ŷ_1), ..., ŷ_m*(1 - ŷ_m))

        4. Newton-Raphson Update Step:
           w_(k+1) = w_k - H^(-1) @ g
        """
        m, n_features = X.shape

        # Add intercept/bias column of 1s to X matrix: X_bias shape becomes (m, n_features + 1)
        X_bias = np.c_[np.ones((m, 1)), X]

        # Initialize weights including bias term
        self.weights = np.zeros(X_bias.shape[1])
        self.loss_history = []

        for i in range(self.n_iters):
            # Step 1: Forward pass (predictions)
            z = np.dot(X_bias, self.weights)
            y_pred = self._sigmoid(z)

            # Step 2: Compute Log Loss
            loss = self._compute_loss(y, y_pred)
            self.loss_history.append(loss)

            # Step 3: Compute Gradient vector g
            gradient = (1 / m) * np.dot(X_bias.T, (y_pred - y))

            # Step 4: Compute Diagonal Variance Matrix S (m x m weights)
            # S_i = ŷ_i * (1 - ŷ_i)
            s_diag = y_pred * (1 - y_pred)

            # Efficient computation of Hessian without generating full (m x m) matrix:
            # H = (1/m) * (X_bias * s_diag[:, None])^T @ X_bias
            Hessian = (1 / m) * np.dot(X_bias.T * s_diag, X_bias)

            # Add small regularizer to Hessian diagonal to guarantee invertibility
            Hessian += self.l2_reg * np.eye(Hessian.shape[0])

            # Step 5: Solve H * Δw = g  =>  Δw = H^(-1) @ g
            # np.linalg.solve is numerically faster and more stable than inverting directly
            step = np.linalg.solve(Hessian, gradient)

            # Step 6: Update Parameters
            self.weights -= step

            # Step 7: Check convergence
            if np.linalg.norm(step) < self.tol:
                print(f"Newton's Method converged at iteration {i + 1}.")
                break

        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        """Predict class probabilities for X."""
        m = X.shape[0]
        X_bias = np.c_[np.ones((m, 1)), X]
        return self._sigmoid(np.dot(X_bias, self.weights))

    def predict(self, X: np.ndarray, threshold: float = 0.5) -> np.ndarray:
        """Predict binary class labels for X."""
        return np.where(self.predict_proba(X) >= threshold, 1, 0)


# =====================================================================
# Example Usage / Test Driver
# =====================================================================
if __name__ == "__main__":
    from sklearn.datasets import make_classification
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import accuracy_score, classification_report

    # 1. Generate Synthetic Dataset
    X, y = make_classification(
        n_samples=1000,
        n_features=5,
        n_classes=2,
        random_state=42
    )

    # 2. Train / Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # 3. Instantiate & Train Model
    clf = LogisticRegressionNewton(n_iters=20, tol=1e-6)
    clf.fit(X_train, y_train)

    # 4. Make Predictions
    predictions = clf.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)

    # 5. Display Results
    print(f"Training Complete!")
    print(f"Final Loss: {clf.loss_history[-1]:.6f}")
    print(f"Iterations Needed: {len(clf.loss_history)}")
    print(f"Test Accuracy: {accuracy * 100:.2f}%\n")
    print("Classification Report:")
    print(classification_report(y_test, predictions))

Newton's Method converged at iteration 7.
Training Complete!
Final Loss: 0.351423
Iterations Needed: 7
Test Accuracy: 88.00%

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.92      0.88        97
           1       0.92      0.84      0.88       103

    accuracy                           0.88       200
   macro avg       0.88      0.88      0.88       200
weighted avg       0.88      0.88      0.88       200

